# GPU Benchmark: Parameter Estimation

This notebook quantifies the **potential** speedup of running Twin4Build parameter
estimation on a GPU. It is designed to run directly in
[Google Colab](https://colab.research.google.com) — select a GPU runtime
(`Runtime > Change runtime type > T4 GPU`) to get the CPU-vs-GPU comparison, or a
CPU runtime to get the CPU baseline only.

## What is measured, and why this way

Twin4Build's tensor code currently runs on CPU only (no device plumbing yet), so a
GPU cannot be benchmarked by flipping a switch on the real pipeline. Instead the
notebook measures two complementary things:

1. **The real estimation pipeline on the current (CPU) runtime.** The bundled
   estimator-example model (single office: thermal + CO2 dynamics, PID + rule-based
   control, 19 parameters, 4 measurement sensors) is calibrated for a few SLSQP
   iterations through the *fast single-shooting* objective — the composed one-step
   map with autodiff gradients. This anchors everything in real seconds-per-evaluation.

2. **The estimator's dominant computational kernels, CPU vs GPU.** The fast
   objective's runtime is dominated by a small set of tensor operations. Each is
   re-implemented below as a compact, self-contained kernel with the *same math and
   the same tensor shapes* as the real code, parameterized by device and batch size:

   - **K1 — bilinear ZOH rollout**: per-step exact discretization via
     `torch.matrix_exp` of the input-augmented system matrix (what
     `bilinear_onestep` does for building-space/wall blocks whose dynamics depend
     on air flows).
   - **K2 — cached-discretization rollout**: sequential `x = Ad x + Bd u` steps plus
     the pointwise controller math (sigmoids, clamps) of the composed one-step map —
     the per-evaluation cost when discretization is reused.
   - **K3 — collocation defect evaluation**: all timesteps evaluated *in parallel*
     (the transcription path), including the backward pass for gradients.

The key variable is the **batch dimension** `B`: how many parameter candidates are
evaluated simultaneously. Today's SLSQP estimation evaluates `B = 1` per step; a GPU
mostly pays off through batching (multi-start, MCMC/ensemble methods, or many-zone
models). The sweep over `B` shows exactly where the crossover is.

In [ ]:
# Colab: install Twin4Build from the git ref baked into this notebook
# (docs/PR badges). Local checkouts skip install. Override with T4B_REF.
from pathlib import Path

_bootstrap = Path("twin4build/examples/colab_bootstrap.py")
if not _bootstrap.is_file():
    _bootstrap = Path("colab_bootstrap.py")

if _bootstrap.is_file():
    exec(_bootstrap.read_text(encoding="utf-8"))
else:
    exec('"""Colab-aware Twin4Build installer (stdlib only - safe before the package exists).\n\nNotebooks opened from GitHub/Colab only fetch the ``.ipynb``; they do not install\nthat git revision. ``pip install twin4build`` hits PyPI (1.x) and is wrong for\ndocs/dev/PR badges.\n\nColab runs cell JS inside an output iframe, so the real\n``/github/.../blob/<ref>/...`` notebook URL is not available to Python. Do **not**\ntry to scrape ``window.location`` / ``document.referrer`` for the git ref.\n\nInstead, install from:\n1. ``T4B_REF`` if set, else\n2. :data:`_T4B_EMBEDDED_REF` (commit SHA baked in by\n   ``scripts/patch_colab_notebooks.py`` when example notebooks are refreshed).\n"""\n\n# Standard library imports\nfrom __future__ import annotations\n\nimport os\nimport re\nimport subprocess\nimport sys\nfrom pathlib import Path\n\n\nREPO_URL = "https://github.com/JBjoernskov/Twin4Build.git"\n_INSTALL_MARKER = Path("/content/.twin4build_colab_ref")\n\n# Branch or commit SHA. Updated by scripts/patch_colab_notebooks.py.\n# Slashy branches are installed via refs/heads/... (see _pip_git_url).\n_T4B_EMBEDDED_REF = "fix/full-workflow-portable-data"\n\n_SHA_RE = re.compile(r"^[0-9a-f]{7,40}$", re.IGNORECASE)\n\n\ndef in_colab() -> bool:\n    return "google.colab" in sys.modules\n\n\ndef _pip_git_url(ref: str) -> str:\n    """Build a pip ``git+`` URL that tolerates slashy branch names."""\n    ref = ref.strip()\n    if not ref:\n        raise ValueError("empty git ref")\n    if _SHA_RE.fullmatch(ref) or ref.startswith("refs/"):\n        spec = ref\n    elif "/" in ref:\n        # ``@fix/foo`` is ambiguous in pip/VCS URLs; use the heads ref.\n        spec = f"refs/heads/{ref}"\n    else:\n        spec = ref\n    return f"git+{REPO_URL}@{spec}"\n\n\ndef detect_git_ref() -> str:\n    """Return ``T4B_REF`` or the baked-in notebook ref."""\n    return os.environ.get("T4B_REF") or _T4B_EMBEDDED_REF\n\n\ndef _import_smoke_ok() -> bool:\n    """Check in a fresh process (avoids half-upgraded in-memory numpy)."""\n    try:\n        subprocess.check_call(\n            [\n                sys.executable,\n                "-c",\n                "import numpy; import twin4build",\n            ],\n            stdout=subprocess.DEVNULL,\n            stderr=subprocess.DEVNULL,\n        )\n        return True\n    except (OSError, subprocess.SubprocessError):\n        return False\n\n\ndef _restart_colab_kernel() -> None:\n    print(\n        "Colab: restarting kernel so numpy/scipy pick up the new install.\\n"\n        "After reconnect, use Runtime > Run all (pip installs persist)."\n    )\n    os.kill(os.getpid(), 9)\n\n\ndef ensure_twin4build():\n    """On Colab, install Twin4Build from git (baked-in SHA / ``T4B_REF``).\n\n    Locally this is a no-op. Returns the ref installed, or ``None`` locally.\n    """\n    if not in_colab():\n        return None\n\n    ref = detect_git_ref()\n    url = _pip_git_url(ref)\n\n    marker_ref = (\n        _INSTALL_MARKER.read_text(encoding="utf-8").strip()\n        if _INSTALL_MARKER.is_file()\n        else None\n    )\n    if marker_ref == ref and _import_smoke_ok():\n        print(f"Colab: twin4build already installed ({url})")\n        return ref\n\n    print(f"Colab: installing twin4build from {url}")\n    subprocess.check_call(\n        [\n            sys.executable,\n            "-m",\n            "pip",\n            "install",\n            "--upgrade",\n            url,\n        ]\n    )\n    try:\n        _INSTALL_MARKER.parent.mkdir(parents=True, exist_ok=True)\n        _INSTALL_MARKER.write_text(ref, encoding="utf-8")\n    except OSError:\n        pass\n\n    if not _import_smoke_ok():\n        print("Colab: install finished but import smoke-test failed.")\n        _restart_colab_kernel()\n        return ref\n\n    if marker_ref != ref:\n        _restart_colab_kernel()\n    return ref\n')

ensure_twin4build()
import twin4build as tb

# --- Setup (Colab-aware) ---------------------------------------------------
# On Colab this installs Twin4Build from the dev branch (a few minutes).
# Locally (repo checked out / package installed) it is a no-op.

import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

DEVICES = ["cpu"] + (["cuda"] if torch.cuda.is_available() else [])
print(f"torch {torch.__version__}")
print(f"CPU threads: {torch.get_num_threads()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU available -- kernel benchmarks will run on CPU only.")
    print("(In Colab: Runtime > Change runtime type > T4 GPU)")


## Part A — Real estimation pipeline on the current (CPU) runtime

The estimator-example model ships with the package (RDF instance graph + sensor
CSVs), so this runs anywhere. We calibrate all 19 parameters against the four
sensors over the example's two 4-day periods (576 timesteps at 20 minutes), limited
to a few SLSQP iterations — enough to time the two phases that matter:

- **setup**: building the composed one-step objective (graph analysis + reference
  rollout capture), paid once per estimation;
- **per-evaluation cost**: one objective value + full gradient via autodiff,
  paid at every solver iterate. This is the number a GPU would have to beat.

In [ ]:
from twin4build.examples.collocation_comparison import (
    EXAMPLE_END,
    EXAMPLE_START,
    STEP_SIZE,
    example_parameters,
    load_model,
)

model = load_model()
simulator = tb.Simulator(model)
estimator = tb.Estimator(simulator)

parameters = example_parameters(model)
measurements = [
    (model.components["office_temperature_sensor"], 0.05),
    (model.components["office_co2_sensor"], 15.0),
    (model.components["office_valve_position_sensor"], 0.025),
    (model.components["office_damper_position_sensor"], 0.025),
]

t0 = time.perf_counter()
result = estimator.estimate(
    EXAMPLE_START,
    EXAMPLE_END,
    STEP_SIZE,
    parameters,
    measurements,
    n_warmup=24,
    method=("scipy", "SLSQP", "ad"),
    options={"maxiter": 5, "fast": True},
)
wall = time.perf_counter() - t0

n_eval = result["nfev"] if "nfev" in result else None
print(f"\nTotal wall time: {wall:.1f} s for {n_eval} objective+gradient evaluations")
if n_eval:
    baseline_s_per_eval = wall / n_eval
    print(f"CPU baseline: {baseline_s_per_eval:.2f} s per evaluation "
          f"(576 timesteps, 19 parameters, B=1)")

## Part B — Kernel benchmarks, CPU vs GPU

Shapes mirror the estimator-example model: the fused office+wall block has
`n_x = 6` states driven by `n_u = 8` inputs, rolled over `T = 288` steps per period
(4 days at 20 minutes). `B` is the number of parameter candidates evaluated
simultaneously — each candidate gets its own system matrices, exactly as a batched
estimator would do it.

Timing methodology: warm-up call first (JIT/cuDNN/allocator), `torch.cuda.synchronize()`
around the timed region, median of repeated runs.

In [ ]:
# --- Benchmark helpers -------------------------------------------------------
N_X, N_U = 6, 8      # fused office+wall block: states / inputs
T = 288              # 4 days at 20-minute steps (one estimation period)
DT = 1200.0
BATCHES = [1, 8, 64, 512, 4096]

def bench(fn, device, repeats=5):
    """Median wall time of fn() with warm-up and CUDA synchronization."""
    fn()  # warm-up
    if device == "cuda":
        torch.cuda.synchronize()
    times = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        fn()
        if device == "cuda":
            torch.cuda.synchronize()
        times.append(time.perf_counter() - t0)
    return float(np.median(times))

def make_system(B, device, seed=0, bilinear=False):
    """B independent stable state-space systems (one per parameter candidate)."""
    g = torch.Generator().manual_seed(seed)
    # Stable continuous-time A: diagonally dominant with negative diagonal,
    # time constants in the hours range like a thermal RC network.
    A = torch.rand(B, N_X, N_X, generator=g) * 1e-5
    A = A - torch.diag_embed(A.sum(-1) + 1.0 / 3600.0)
    Bm = torch.rand(B, N_X, N_U, generator=g) * 1e-4
    u = torch.rand(B, T, N_U, generator=g)
    x0 = torch.rand(B, N_X, generator=g)
    out = [A.to(device), Bm.to(device), u.to(device), x0.to(device)]
    if bilinear:
        # One bilinear channel (air flow scaling the loss term), as in the
        # building-space mass balance and wall coupling.
        E = torch.rand(B, N_X, N_X, generator=g) * 1e-5
        out.append(E.to(device))
    return out

def run_table(name, runner, batches=BATCHES):
    """Time `runner(B, device)` over batches x devices; return a tidy table."""
    rows = []
    for B in batches:
        row = {"B": B}
        for dev in DEVICES:
            row[dev] = bench(lambda: runner(B, dev), dev)
        if "cuda" in row:
            row["speedup"] = row["cpu"] / row["cuda"]
        rows.append(row)
    df = pd.DataFrame(rows).set_index("B")
    print(f"\n=== {name} (seconds per call) ===")
    print(df.to_string(float_format=lambda v: f"{v:.4f}"))
    return df

### K1 — Bilinear ZOH rollout (per-step `matrix_exp`)

Building-space and wall dynamics are *bilinear*: the continuous-time matrices depend
on inputs (air flow rates), so the exact zero-order-hold discretization must be
recomputed **every step** — a `matrix_exp` of the input-augmented
`(n_x + n_u) x (n_x + n_u)` matrix per step per candidate. This is the single most
expensive operation in the fast objective (it showed up at the top of the real
pipeline's profile). To keep the benchmark quick we time a 24-step slice and report
the *per-step* cost, which is what scales with horizon length.

In [ ]:
T_SLICE = 24  # timed steps; report per-step cost

def k1_bilinear_rollout(B, device):
    A, Bm, u, x0, E = make_system(B, device, bilinear=True)
    n_aug = N_X + N_U
    x = x0
    for t in range(T_SLICE):
        u_t = u[:, t]                                   # (B, n_u)
        # Input-dependent drift: A + u_flow * E  (bilinear term)
        A_t = A + u_t[:, :1, None] * E                  # (B, n_x, n_x)
        M = torch.zeros(B, n_aug, n_aug, device=device)
        M[:, :N_X, :N_X] = A_t
        M[:, :N_X, N_X:] = Bm
        Phi = torch.matrix_exp(M * DT)                  # exact ZOH
        x = torch.einsum("bij,bj->bi", Phi[:, :N_X, :N_X], x) + torch.einsum(
            "bij,bj->bi", Phi[:, :N_X, N_X:], u_t
        )
    return x

df_k1 = run_table("K1: bilinear ZOH rollout", k1_bilinear_rollout)
df_k1_per_step = df_k1 / T_SLICE
print(f"\nPer-step cost (s); full period would be x{T} steps")

### K2 — Cached-discretization rollout with controller math

When the discrete matrices can be reused (linear blocks, or between input changes),
one composed step is a small matrix-vector product plus the pointwise algebra of the
controllers (PID with anti-windup clamps, sigmoid occupancy gate, damper
exponentials). This kernel rolls the **full period** (`T = 288` sequential steps)
including that pointwise math — it is the shape of the fast objective's forward
pass, and its sequential nature is exactly what limits GPU utilization at `B = 1`.

In [ ]:
def k2_sequential_rollout(B, device):
    A, Bm, u, x0 = make_system(B, device)
    Ad = torch.matrix_exp(A * DT)                       # cached once
    Bd = torch.einsum("bij,bjk->bik", Ad - torch.eye(N_X, device=device), 
                      torch.linalg.solve(A, Bm))
    x = x0
    integral = torch.zeros(B, device=device)
    for t in range(T):
        u_t = u[:, t]
        # PID-ish pointwise controller math (composed one-step map contains
        # several of these per step: PID, on/off, sigmoid gate, damper exp)
        err = 21.0 - x[:, 0]
        integral = torch.clamp(integral + err * DT / 3600.0, 0.0, 1.0)
        ctrl = torch.sigmoid(30.0 * (err + integral))
        damper = 0.2 + 0.8 * (torch.exp(3.4 * ctrl) - 1.0) / (np.exp(3.4) - 1.0)
        u_eff = torch.cat([u_t[:, :-1], damper[:, None]], dim=-1)
        x = torch.einsum("bij,bj->bi", Ad, x) + torch.einsum(
            "bij,bj->bi", Bd, u_eff
        )
    return x

df_k2 = run_table("K2: sequential rollout, T=288 (one period)", k2_sequential_rollout)

### K3 — Collocation defect evaluation (parallel in time) + gradient

The collocation transcription evaluates the continuity defects
`r_t = x_{t+1} - (Ad x_t + Bd u_t)` for **all timesteps at once** — no sequential
loop — plus a backward pass for the gradient of the defect norm w.r.t. the state
trajectory and parameters. This is the estimation path with by far the best GPU
fit: one big batched matmul over `(B, T, n_x)` instead of `T` small sequential ones.

In [ ]:
def k3_collocation_defects(B, device):
    A, Bm, u, x0 = make_system(B, device)
    Ad = torch.matrix_exp(A * DT)
    Bd = torch.einsum("bij,bjk->bik", Ad - torch.eye(N_X, device=device),
                      torch.linalg.solve(A, Bm))
    # Decision variables: the full state trajectory (as in the NLP)
    xs = torch.rand(B, T + 1, N_X, device=device, requires_grad=True)
    pred = torch.einsum("bij,btj->bti", Ad, xs[:, :-1]) + torch.einsum(
        "bij,btj->bti", Bd, u
    )
    defects = xs[:, 1:] - pred                          # (B, T, n_x), all at once
    loss = (defects**2).sum()
    loss.backward()                                     # gradient wrt trajectory
    return xs.grad

df_k3 = run_table("K3: collocation defects + backward (T=288 parallel)",
                  k3_collocation_defects)

In [ ]:
# --- Summary plot ------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=False)
for ax, (df, title) in zip(
    axes,
    [
        (df_k1, "K1: bilinear ZOH rollout"),
        (df_k2, "K2: sequential rollout"),
        (df_k3, "K3: collocation defects"),
    ],
):
    for dev in DEVICES:
        # Normalize to per-candidate cost: lower = cheaper per parameter set
        ax.loglog(df.index, df[dev] / df.index, "o-", label=dev)
    ax.set_title(title)
    ax.set_xlabel("batch size B (parameter candidates)")
    ax.grid(True, which="both", alpha=0.3)
axes[0].set_ylabel("seconds per candidate")
axes[0].legend()
plt.tight_layout()
plt.show()

if "cuda" in DEVICES:
    print("GPU speedup at largest batch:")
    for name, df in [("K1", df_k1), ("K2", df_k2), ("K3", df_k3)]:
        print(f"  {name}: {df['speedup'].iloc[-1]:.1f}x "
              f"(B=1: {df['speedup'].iloc[0]:.2f}x)")

## How to read the results

Typical findings on a Colab T4 (your numbers above will vary):

- **At `B = 1` — today's SLSQP estimation — the GPU is *slower* than the CPU.**
  Every step of a 6-state model is a microsecond-scale operation; kernel-launch
  latency (~5–10 µs per op) dominates, and the sequential dependency between steps
  prevents any parallelism in time. Porting the current single-candidate estimation
  loop to GPU as-is would be a net loss.

- **The per-candidate cost on GPU falls almost linearly with `B`** until the device
  saturates (typically `B` in the hundreds–thousands for these sizes). At `B = 4096`,
  kernels K1/K3 usually land at one to two orders of magnitude speedup per candidate.

- **K3 (collocation) benefits most**: it is parallel in *time* as well as batch, so
  it saturates the GPU even at small `B`.

**What this means for Twin4Build:**

1. GPU pays off for *population-based* workflows: multi-start estimation, MCMC /
   ensemble UQ, scenario studies, or estimating across many zones/buildings at once
   (the existing `n_c` component-batching maps directly onto `B`).
2. The collocation transcription is the natural first target for a GPU port —
   parallel-in-time defect evaluation, no sequential rollout in the inner loop.
3. Making it real requires device plumbing through the framework: a `device`
   argument on `Model`/`Simulator`, tensors and interpolation tables created on the
   target device, and `.cpu()` bridges at the numpy boundaries (scipy solvers,
   plotting, CSV I/O).